# GNN ablations: what does the graph actually contribute?

The analysis of saved predictions could not say *why* the GNN wins on some classes and loses on others. This notebook answers it with **controlled experiments**: the same GNN is retrained with exactly one thing changed.

| Condition | What changes | What it tells us |
|---|---|---|
| **no graph** | node embeddings are zeroed, the classifier sees only the flow's own features | the GNN's classifier head alone, same training recipe |
| **shuffled edges** | every flow keeps its features and label but gets another flow's endpoints (train and test graphs); node degrees stay the same | whether the *real* connections matter, or just having a graph-shaped input |
| **real graph, small / default / large neighbourhood** | neighbours sampled per node: 2/2, 8/5 (the original), 16/10 | whether more context helps |

Reading the results: `real - shuffled` = value of the true topology. `shuffled - no graph` = what a meaningless graph adds (usually noise). If `real` is not clearly above `shuffled`, the connections carry no usable information beyond the flow's own features.

Every condition is repeated with **3 seeds** (seed changes the validation split, weight initialisation and neighbour sampling; the same seeds are used for every condition, so the comparison is paired). Test data is never used for training or early stopping.

In [ ]:
import copy
import glob
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

DEBUG = False   # True = quick smoke test on a small random sample
SEEDS = [42] if DEBUG else [42, 43, 44]
DATASET_PATH = "/kaggle/input/datasets/shahiismyname/unsw-nb15-preprocessed-dataset"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

HIDDEN = 64
BATCH = 2048
MAX_EPOCHS = 2 if DEBUG else 12
PATIENCE = 3
LR = 2e-3
print("device:", DEVICE, "| debug:", DEBUG, "| seeds:", SEEDS)

_fig_count = 0
_plt_show = plt.show


def show_and_save(*args, **kwargs):
    """Show every figure and also keep it as a PNG in the notebook output."""
    global _fig_count
    _fig_count += 1
    plt.savefig(f"/kaggle/working/fig_{_fig_count:02d}.png", dpi=130, bbox_inches="tight")
    _plt_show(*args, **kwargs)


plt.show = show_and_save

## 1. Load data and define the graph

Same data and graph construction as the GNN notebook: nodes are (IP, port) endpoints, edges are flows, and the train graph and test graph are built separately (inductive setting).

In [ ]:
train_df = pd.read_parquet(f"{DATASET_PATH}/train.parquet")
test_df = pd.read_parquet(f"{DATASET_PATH}/test.parquet")
if DEBUG:
    train_df, test_df = train_df.sample(150_000, random_state=0), test_df.sample(30_000, random_state=0)
train_graph = pd.read_csv(f"{DATASET_PATH}/train_graph.csv", index_col="row_id").loc[train_df.index]
test_graph = pd.read_csv(f"{DATASET_PATH}/test_graph.csv", index_col="row_id").loc[test_df.index]

class_names = joblib.load(f"{DATASET_PATH}/preprocessing.pkl")["label_encoder"].classes_
n_classes = len(class_names)
feature_cols = [c for c in train_df.columns if c != "label"]
print("train edges", len(train_df), "| test edges", len(test_df), "| edge features", len(feature_cols))


def build_graph(graph_df, feat_df):
    n = len(graph_df)
    ip_code, _ = pd.factorize(pd.concat([graph_df["srcip"], graph_df["dstip"]]))
    ip_code = ip_code.astype("int64")
    ports = np.concatenate([graph_df["sport"].fillna(-1), graph_df["dsport"].fillna(-1)]).astype("int64") + 1
    node_id, node_keys = pd.factorize(ip_code * 70_000 + ports)   # one node per distinct (IP, port)
    src, dst = torch.from_numpy(node_id[:n]), torch.from_numpy(node_id[n:])
    n_nodes = len(node_keys)

    nodes = torch.cat([src, dst])          # every edge appears once per endpoint
    other = torch.cat([dst, src])
    edge_ids = torch.cat([torch.arange(n), torch.arange(n)])
    order = torch.argsort(nodes)
    ptr = torch.zeros(n_nodes + 1, dtype=torch.long)
    ptr[1:] = torch.cumsum(torch.bincount(nodes, minlength=n_nodes), 0)
    G = {"n_nodes": n_nodes, "src": src, "dst": dst, "ptr": ptr,
         "inc_edge": edge_ids[order], "inc_other": other[order],
         "x": torch.from_numpy(feat_df[feature_cols].to_numpy("float32")),
         "y": torch.from_numpy(feat_df["label"].to_numpy("int64"))}
    return {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in G.items()}


def shuffle_endpoints(graph_df, seed):
    """Give every flow another flow's endpoints (the four columns move together).
    Flow features and labels stay put; the number of flows at each endpoint is unchanged."""
    perm = np.random.default_rng(seed).permutation(len(graph_df))
    out = graph_df.copy()
    for col in ["srcip", "dstip", "sport", "dsport"]:
        out[col] = graph_df[col].to_numpy()[perm]
    return out


def sample_neighbors(G, nodes, fanout):
    """Draw `fanout` incident edges (with replacement) for each node -> (edge ids, neighbour node ids)."""
    start = G["ptr"][nodes]
    deg = G["ptr"][nodes + 1] - start
    offset = (torch.rand(len(nodes), fanout, device=nodes.device) * deg.unsqueeze(1)).long()
    offset = torch.minimum(offset, deg.unsqueeze(1) - 1)
    pos = start.unsqueeze(1) + offset
    return G["inc_edge"][pos], G["inc_other"][pos]


G_train_real, G_test_real = build_graph(train_graph, train_df), build_graph(test_graph, test_df)

## 2. Model

The E-GraphSAGE from the GNN notebook, with two switches: the neighbourhood sizes, and `use_graph=False`, which replaces both node embeddings by zeros so the classifier only sees the flow's own features.

In [ ]:
class EdgeSAGE(nn.Module):
    def __init__(self, n_feat, hidden, n_classes, fanout_2hop=8, fanout_1hop=5, use_graph=True):
        super().__init__()
        self.fanout_2hop, self.fanout_1hop, self.use_graph = fanout_2hop, fanout_1hop, use_graph
        self.edge1 = nn.Linear(n_feat, hidden)
        self.upd1 = nn.Linear(1 + hidden, hidden)
        self.node2 = nn.Linear(hidden, hidden, bias=False)
        self.edge2 = nn.Linear(n_feat, hidden)
        self.upd2 = nn.Linear(2 * hidden, hidden)
        self.head = nn.Sequential(nn.Linear(2 * hidden + n_feat, 128), nn.ReLU(),
                                  nn.Dropout(0.2), nn.Linear(128, n_classes))

    def layer1(self, G, nodes):
        eids, _ = sample_neighbors(G, nodes, self.fanout_1hop)
        msg = torch.relu(self.edge1(G["x"][eids]))
        ones = torch.ones(len(nodes), 1, device=nodes.device)             # constant initial node feature
        return torch.relu(self.upd1(torch.cat([ones, msg.mean(1)], dim=1)))

    def forward(self, G, idx):
        if not self.use_graph:
            zeros = torch.zeros(len(idx), self.head[0].in_features - G["x"].shape[1], device=idx.device)
            return self.head(torch.cat([zeros, G["x"][idx]], dim=1))
        ends = torch.cat([G["src"][idx], G["dst"][idx]])
        h1_self = self.layer1(G, ends)
        eids, nbrs = sample_neighbors(G, ends, self.fanout_2hop)
        h1_nbr = self.layer1(G, nbrs.reshape(-1)).view(len(ends), self.fanout_2hop, -1)
        msg = torch.relu(self.node2(h1_nbr) + self.edge2(G["x"][eids]))
        h2 = torch.relu(self.upd2(torch.cat([h1_self, msg.mean(1)], dim=1)))
        return self.head(torch.cat([h2[:len(idx)], h2[len(idx):], G["x"][idx]], dim=1))


@torch.no_grad()
def predict_proba(model, G, idx, passes=1, batch=8192):
    model.eval()
    out = []
    for i in range(0, len(idx), batch):
        b = idx[i:i + batch]
        out.append((sum(torch.softmax(model(G, b), 1) for _ in range(passes)) / passes).cpu())
    return torch.cat(out)

## 3. One run = one condition and one seed

Training follows the GNN notebook: 15% of the train edges are held out for early stopping on validation macro F1, square-root balanced class weights, Adam. The test graph is only used at the end, with 5 neighbour-sampling passes.

In [ ]:
def run(name, seed, shuffle=False, use_graph=True, fanouts=(8, 5)):
    t0 = time.time()
    torch.manual_seed(seed)
    np.random.seed(seed)
    if shuffle:
        G_tr = build_graph(shuffle_endpoints(train_graph, seed), train_df)
        G_te = build_graph(shuffle_endpoints(test_graph, seed + 1000), test_df)
    else:
        G_tr, G_te = G_train_real, G_test_real

    y_all = G_tr["y"].cpu().numpy()
    tr_np, va_np = train_test_split(np.arange(len(y_all)), test_size=0.15, stratify=y_all, random_state=seed)
    tr_idx, va_idx = torch.from_numpy(tr_np).to(DEVICE), torch.from_numpy(va_np).to(DEVICE)
    cw = np.sqrt(compute_class_weight("balanced", classes=np.arange(n_classes), y=y_all[tr_np]))
    class_weights = torch.tensor(cw, dtype=torch.float32, device=DEVICE)

    model = EdgeSAGE(len(feature_cols), HIDDEN, n_classes, fanouts[0], fanouts[1], use_graph).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    y_val = G_tr["y"][va_idx].cpu().numpy()

    best, bad_epochs = {"f1": -1.0, "state": None, "epoch": 0}, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        perm = tr_idx[torch.randperm(len(tr_idx), device=DEVICE)]
        for i in range(0, len(perm), BATCH):
            b = perm[i:i + BATCH]
            opt.zero_grad()
            loss_fn(model(G_tr, b), G_tr["y"][b]).backward()
            opt.step()
        val_f1 = f1_score(y_val, predict_proba(model, G_tr, va_idx).argmax(1).numpy(), average="macro")
        if val_f1 > best["f1"]:
            best, bad_epochs = {"f1": val_f1, "state": copy.deepcopy(model.state_dict()), "epoch": epoch}, 0
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                break

    model.load_state_dict(best["state"])
    te_idx = torch.arange(len(G_te["src"]), device=DEVICE)
    pred = predict_proba(model, G_te, te_idx, passes=5).argmax(1).numpy()
    y_te = G_te["y"].cpu().numpy()
    per_class = f1_score(y_te, pred, labels=np.arange(n_classes), average=None, zero_division=0)
    print(f"{name:<34} seed {seed} | best epoch {best['epoch']:>2} | val F1 {best['f1']:.4f} | "
          f"test macro F1 {per_class.mean():.4f} | {time.time() - t0:.0f}s", flush=True)
    return {"condition": name, "seed": seed, "val macro F1": best["f1"], "test macro F1": per_class.mean(),
            **{f"F1 {c}": f for c, f in zip(class_names, per_class)}}


CONDITIONS = {
    "no graph (own features only)": dict(use_graph=False),
    "shuffled edges (8/5)": dict(shuffle=True),
    "real graph, small (2/2)": dict(fanouts=(2, 2)),
    "real graph, default (8/5)": dict(),
    "real graph, large (16/10)": dict(fanouts=(16, 10)),
}

## 4. Run all conditions

In [ ]:
rows = []
for seed in SEEDS:
    for name, kwargs in CONDITIONS.items():
        rows.append(run(name, seed, **kwargs))
results = pd.DataFrame(rows)
results.to_csv("/kaggle/working/gnn_ablation_runs.csv", index=False)

## 5. Results

Bars = mean over seeds, dots = individual seeds, dashed line = the tabular two-stage LightGBM from the baseline notebook (a different model family, shown for reference).

In [ ]:
tab_file = glob.glob("/kaggle/input/**/two_stage_test_proba.parquet", recursive=True)
tab_f1 = None
if tab_file and not DEBUG:
    tab = pd.read_parquet(tab_file[0]).loc[test_df.index]
    tab_f1 = f1_score(test_df["label"].to_numpy(), tab.to_numpy().argmax(1), average="macro")

summary = results.groupby("condition", sort=False)[["val macro F1", "test macro F1"]].agg(["mean", "std"])
print(summary.round(4).to_string())
if tab_f1 is not None:
    print("tabular two-stage LightGBM macro F1: %.4f" % tab_f1)

fig, ax = plt.subplots(figsize=(10, 4.5))
order = list(CONDITIONS)
sns.barplot(data=results, y="condition", x="test macro F1", order=order, errorbar="sd", capsize=0.15, color="steelblue", alpha=0.8, ax=ax)
sns.stripplot(data=results, y="condition", x="test macro F1", order=order, color="black", size=5, ax=ax)
if tab_f1 is not None:
    ax.axvline(tab_f1, color="darkorange", ls="--", label=f"tabular two-stage LightGBM ({tab_f1:.3f})")
    ax.legend(loc="lower right")
ax.set_xlim(0.3, 0.7)
ax.set_title("Test macro F1 by condition (mean and std over seeds)")
plt.tight_layout()
plt.show()

## 6. Per class: what does the real graph add over a shuffled one?

Left: mean test F1 per class for every condition. Right: `real graph - shuffled edges` per class, with the seed standard deviation of the difference as error bars. Bars whose error bars cross zero are not distinguishable from noise.

In [ ]:
class_cols = [f"F1 {c}" for c in class_names]
mean_pc = results.groupby("condition", sort=False)[class_cols].mean()
mean_pc.columns = class_names
print(mean_pc.loc[order].round(3).T.to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 4.8), gridspec_kw={"width_ratios": [3, 2]})
sns.heatmap(mean_pc.loc[order].T, annot=True, fmt=".2f", cmap="viridis", cbar=False, ax=axes[0])
axes[0].set_title("Mean test F1 per class")
axes[0].set_ylabel("")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=20)

real = results[results.condition == "real graph, default (8/5)"].set_index("seed")[class_cols]
shuf = results[results.condition == "shuffled edges (8/5)"].set_index("seed")[class_cols]
diff = (real - shuf)
diff.columns = class_names
diff_mean, diff_std = diff.mean(), diff.std()
pd.DataFrame({"real - shuffled": diff_mean}).plot.barh(ax=axes[1], xerr=diff_std, color="gray", legend=False, capsize=3)
axes[1].axvline(0, color="black", lw=0.8)
axes[1].set_title("Value of the real connections (F1 difference)")
plt.tight_layout()
plt.show()
print(pd.DataFrame({"mean": diff_mean, "std": diff_std}).round(3).to_string())